<a href="https://colab.research.google.com/github/edgarbonet/XatBot_1.4_EdgarBonet/blob/main/Backend_XatBot_WordPress.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flask-cors flask google-genai pyngrok beautifulsoup4 requests --quiet

import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from flask import Flask, request, jsonify
from flask_cors import CORS
from google import genai
from google.colab import userdata
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)

# --- CONFIGURACIÓ ---
API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=API_KEY)

START_URL = "https://ebonet.inscastellbisbal.net/"
MAX_PAGES = 99  # Quantes subpàgines vols que llegeixi com a màxim

def get_all_links(url, domain):
    """Busca tots els enllaços interns d'una pàgina"""
    links = set()
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        res = requests.get(url, headers=headers, timeout=10, verify=False)
        soup = BeautifulSoup(res.text, 'html.parser')
        for a in soup.find_all('a', href=True):
            link = urljoin(url, a['href'])
            # Només afegim si és del mateix domini i és una web (no un PDF o imatge)
            if domain in link and not any(ext in link.lower() for ext in ['.pdf', '.jpg', '.png', '.zip']):
                # Treiem l'àncora (#) per no repetir pàgines
                links.add(link.split('#')[0])
    except Exception as e:
        print(f"Error buscant enllaços a {url}: {e}")
    return links

def crawl_website(start_url):
    """Visita les pàgines una per una i extreu el text"""
    domain = urlparse(start_url).netloc
    to_visit = {start_url}
    visited = set()
    all_text = ""

    print(f"🚀 Iniciant Crawler a {start_url}...")

    while to_visit and len(visited) < MAX_PAGES:
        url = to_visit.pop()
        if url in visited: continue

        try:
            print(f"📄 Llegint ({len(visited)+1}/{MAX_PAGES}): {url}")
            headers = {'User-Agent': 'Mozilla/5.0'}
            res = requests.get(url, headers=headers, timeout=10, verify=False)
            visited.add(url)

            if res.status_code == 200:
                soup = BeautifulSoup(res.text, 'html.parser')

                # Trobar nous enllaços per a la següent iteració
                new_links = get_all_links(url, domain)
                to_visit.update(new_links - visited)

                # Netejar i extreure text
                for s in soup(['script', 'style', 'nav', 'footer', 'header']):
                    s.decompose()

                page_text = ' '.join(soup.get_text(separator=' ').split())
                all_text += f"\n--- PÀGINA: {url} ---\n{page_text}\n"

        except Exception as e:
            print(f"❌ Error a {url}: {e}")

    print(f"✅ Crawler finalitzat. Pàgines llegides: {len(visited)}")
    return all_text[:30000] # Límit de caràcters per al prompt

# Executem el crawler en encendre el servidor
contingut_ebonet = crawl_website(START_URL)

system_prompt = f"""
Ets l'assistent virtual del prtafolis digital de l'estudiant Edgar Bonet.
Tens accés a tota aquesta informació extreta de la web oficial:
{contingut_ebonet}

Instruccions:
1. Respon preguntes sobre qualsevol secció de la web que hagis llegit.
2. Si no saps la resposta perquè no estava a les pàgines llegides, suggereix a l'usuari que consulti secretaria.
3. Respon sempre en català de forma amable.
"""

def start_new_chat():
    return client.chats.create(
        model="gemini-2.5-flash",
        config=genai.types.GenerateContentConfig(system_instruction=system_prompt)
    )

chat_session = start_new_chat()

@app.route('/chat', methods=['POST'])
def handle_chat():
    global chat_session
    try:
        data = request.json
        user_msg = data.get("message", "")
        response = chat_session.send_message(user_msg)
        return jsonify({"reply": response.text.strip()})
    except:
        return jsonify({"reply": "Error de connexió interna."})

!pkill -f ngrok
time.sleep(1)
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
public_url = ngrok.connect(5000).public_url
print(f"\n🔗 URL PER WORDPRESS: {public_url}")
app.run(port=5000)

🚀 Iniciant Crawler a https://ebonet.inscastellbisbal.net/...
📄 Llegint (1/99): https://ebonet.inscastellbisbal.net/


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ebonet.inscastellbisbal.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ebonet.inscastellbisbal.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


📄 Llegint (2/99): https://ebonet.inscastellbisbal.net/2026/02/02/redactar-el-pla-demergencia-i-evacuacio-de-lespai-de-la-lan-party/


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ebonet.inscastellbisbal.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ebonet.inscastellbisbal.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


📄 Llegint (3/99): https://ebonet.inscastellbisbal.net/2025/09/29/utilitzacio-de-marcadors-en-documents-google/


...

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ebonet.inscastellbisbal.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ebonet.inscastellbisbal.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


📄 Llegint (99/99): https://ebonet.inscastellbisbal.net/2025/09/18/apunts-seo-i-sem/


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ebonet.inscastellbisbal.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ebonet.inscastellbisbal.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✅ Crawler finalitzat. Pàgines llegides: 99

🔗 URL PER WORDPRESS: https://postconsonantal-paityn-remonstratingly.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
